# SPADE: CPU vs Multi-GPU Benchmarking

This notebook demonstrates how to utilize SPADE's new GPU backend using `autodock_gpu`. It compares a classic single-snapshot approach against a full 10-conformer NMA breathing ensemble, tracking how SPADE round-robins the heavy workloads across all available GPU devices (like Kaggle's T4 x2).

### 1. Environment Setup

In [ ]:
!apt-get update && apt-get install -y swig
!conda install -c conda-forge autodock-gpu autogrid rdkit prody vina prolif numba -y
!pip install gemmi pandas
!pip install --upgrade git+https://github.com/ChandraguptSharma07/Spade.git

### 2. Prepare the Target & Ligands
We are using EGFR (P00533) as our target against two known drugs (Erlotinib, Gefitinib) and one decoy (Ibuprofen).

In [ ]:
import time
import numpy as np
import pandas as pd
from spade.core.structure import fetch_structure
from spade.core.ligand import prepare_ligand
from spade.core.docking import compute_bounding_box, get_docking_engine
from spade.core.ensemble import EnsembleGenerator
from spade.core.flexibility import build_flexibility_graph
from spade.core.clustering import cluster_poses

print("Fetching AF2 Structure (P00533)...")
structure = fetch_structure('P00533')
rigid_conformer = structure.atoms

# Pre-defined pocket roughly covering the EGFR ATP loop
pocket_residues = np.arange(695, 720)

smiles_dict = {
    "Erlotinib (Active)": "CCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOCCO",
    "Gefitinib (Active)": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN3CCOCC3",
    "Ibuprofen (Decoy)": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
}

ligand_vault = {}
print("Preparing Ligands...")
for name, smi in smiles_dict.items():
    variants = prepare_ligand(smi, ph=7.4, enumerate_stereo=False)
    ligand_vault[name] = variants[0]

# Create base bounding box for the rigid state
bbox = compute_bounding_box(rigid_conformer, pocket_residues)

### 3. Phase 1: Classical Rigid Docking (Single GPU)

In [ ]:
classical_results = []

print("\n--- Running Classical Single Rigid Docking (AD4 Backend) ---")
engine = get_docking_engine(backend="gpu", exhaustiveness=8, device_id=0, n_runs=24)

for name, lig in ligand_vault.items():
    t0 = time.perf_counter()
    poses = engine.dock(
        conformer=rigid_conformer, 
        ligand=lig,
        bbox=bbox,
        conf_idx=0
    )
    t_elapsed = time.perf_counter() - t0
    
    # Note: AD4 scoring scale differs slightly from empirical Vina kcal/mol
    top_score = poses[0].score_kcal_mol if poses else 0.0
    print(f"{name}: {top_score:.2f} kcal/mol ({t_elapsed:.1f}s)")
    classical_results.append((name, top_score, t_elapsed))

### 4. Phase 2: SPADE Ensemble Docking (Multi-GPU Parallelization)
This uses our updated Normal Mode Analysis fixes to generate 10 breathing models, then seamlessly uses `ThreadPoolExecutor` internally inside SPADE to fire off all tasks asynchronously.

In [ ]:
from spade.core.docking import dock_ensemble

print("\n--- Generating 10-conformer Breathing Ensemble ---")
profile = build_flexibility_graph(structure, pocket_residues, cutoff_angstrom=12.0)
gen = EnsembleGenerator(structure, profile, n_conformers=10)
conformers = gen.generate()

spade_results = []

for name, lig in ligand_vault.items():
    print(f"\nEnsemble Docking: {name}...")
    t0 = time.perf_counter()
    
    # SPADE auto-detects Kaggle's GPUs (e.g. device_ids=[0,1]) and round-robins the 10 conformers
    ensemble_out = dock_ensemble(
        conformers,
        [lig],            # Docking just one ligand variant at a time for this loop
        pocket_residues,
        exhaustiveness=8,
        backend="gpu"     # Explicitly ask for AutoDock-GPU
    )
    t_elapsed = time.perf_counter() - t0
    
    print("Clustering consensus...")
    consensus = cluster_poses(ensemble_out, conformers, lig.mol)
    
    spade_results.append((name, consensus.top_cluster_score, consensus.fraction_ensemble, t_elapsed))

### 5. Final Benchmarks

In [ ]:
data = []
for c, s in zip(classical_results, spade_results):
    # c is (name, score, time) | s is (name, score, fraction, time)
    name = c[0]
    data.append({
        "Ligand": name,
        "Classical Score (AD4)": round(c[1], 2),
        "Classical Time (s)": round(c[2], 1),
        "SPADE Consensus": round(s[1], 2) if s[2] > 0 else "N/A",
        "Ensemble Coverage": f"{s[2]*100:.0f}%",
        "Multi-GPU Time (s)": round(s[3], 1),
        "Time Cost Multiplier": round(s[3] / c[2], 1) # Expected ~5x instead of 10x due to 2 GPUs
    })

df = pd.DataFrame(data)

# Save locally
df.to_csv("gpu_benchmark_results.csv", index=False)

display(df)

### 6. Download Results

In [ ]:
from IPython.display import FileLink, display

print("Click to download the benchmark csv:")
display(FileLink("gpu_benchmark_results.csv"))